In [54]:
import os
from dotenv import load_dotenv
load_dotenv() 

True

In [55]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"] = os.getenv("LANGSMITH_TRACING")
os.environ["LANGSMITH_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = os.getenv("LANGSMITH_PROJECT")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [56]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings 
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) 


## Simple AI Assistant

In [57]:
while True:
    question = input("Enter a question: ")
    if question != "quit":
        print(llm.invoke(question).content)
    else:
        print("Exiting...")
        break

Exiting...


In [58]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory 
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

In [59]:
store = {}

In [60]:
def get_session_id(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory() 
    return store[session_id] 

In [61]:
config = {"configurable": {"session_id": "firstchat"}}

In [62]:
model_with_memory = RunnableWithMessageHistory(llm, get_session_id)

In [63]:
model_with_memory.invoke(("Hi, I am Ankit"),config=config).content

'Hi Ankit! How can I assist you today?'

In [64]:
model_with_memory.invoke(("What is my name?"),config=config).content

'Your name is Ankit. How can I help you today?'

## RAG with LCEL

In [65]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma 
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

In [66]:

loader = DirectoryLoader("data/", glob="*.txt", loader_cls=TextLoader)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50, 
    chunk_overlap=10,
    length_function=len)

new_docs = text_splitter.split_documents(documents=docs)
doc_strings = [doc.page_content for doc in new_docs]


db = Chroma.from_documents(
    new_docs,
    embeddings)

retriever = db.as_retriever(search_kwargs={"k": 4})




In [67]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template) 

In [68]:
retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | llm
    | StrOutputParser()
)

In [69]:
question = "What is Llama3? Can you highlight 3 important points?"
print(retrieval_chain.invoke(question))

Llama 3 is a model released by Meta. Here are three important points about it:

1. **Release Date**: Llama 3 is set to be released in April 2024.
2. **Developer**: It is developed by Meta, indicating it is part of their ongoing efforts in AI and machine learning.
3. **Additional Features**: Alongside the release, Meta has added new features or enhancements, although specific details about these features are not provided in the context.


## Lets start with Tools and Agents

In [70]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper 

In [71]:
api_wrapper = WikipediaAPIWrapper() 

In [72]:
tool = WikipediaQueryRun(api_wrapper=api_wrapper) 

In [73]:
tool.name 

'wikipedia'

In [74]:
tool.description

'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.'

In [75]:
tool.args 

{'query': {'description': 'query to look up on wikipedia',
  'title': 'Query',
  'type': 'string'}}

In [76]:
print(tool.run({"query": "Ratan Tata"}))

Page: Ratan Tata
Summary: Ratan Naval Tata (28 December 1937 – 9 October 2024) was an Indian industrialist and philanthropist. He served as the chairman of Tata Group and Tata Sons from 1991 to 2012 and he held the position of interim chairman from October 2016 to February 2017. In 2000, he received the Padma Bhushan, the third highest civilian honour in India, followed by the Padma Vibhushan, the country's second highest civilian honour, in 2008.
Ratan Tata was the son of Naval Tata, who was adopted by Ratanji Tata, son of Jamshedji Tata, the founder of the Tata Group. He graduated from Cornell University College of Architecture with a bachelor's degree in architecture. He had also attended the Harvard Business School (HBS) Advanced Management Program in 1975. He joined the Tata Group in 1962, starting on the shop floor of Tata Steel. He later succeeded J. R. D. Tata as chairman of Tata Sons upon the latter's retirement in 1991. During his tenure, the Tata Group acquired Tetley, Jagua

In [77]:
from langchain_community.tools import YouTubeSearchTool

In [78]:
tool2 = YouTubeSearchTool()

In [79]:
tool2.name

'youtube_search'

In [80]:
tool2.run("Lakshya Chaudhary")

"['https://www.youtube.com/watch?v=lcQaxFSi4ME&pp=ygURTGFrc2h5YSBDaGF1ZGhhcnk%3D', 'https://www.youtube.com/watch?v=dYBFf8ckhnk&pp=ygURTGFrc2h5YSBDaGF1ZGhhcnk%3D']"

In [81]:
from langchain_tavily import TavilySearch

In [82]:
tool3 = TavilySearch()

In [83]:
tool3.invoke({"query": "What happened in Venizvule yesterday?"})

{'query': 'What happened in Venizvule yesterday?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.cbsnews.com/tag/venezuela/',
   'title': "Venezuela news - Today's latest updates",
   'content': "Photos show damage at Fuerte Tiuna, the military base were Maduro was captured · U.S. seeks to tap Venezuela's vast oil reserves after military strikes.",
   'score': 0.22535734,
   'raw_content': None},
  {'url': 'https://www.youtube.com/watch?v=E-XvHEBuSLQ',
   'title': 'Venezuela declares a state of emergency after explosions around ...',
   'content': 'US officials confirmed to several media outlets that the White House is conducting strikes in Venezuela. The Venezuelan government condemned',
   'score': 0.18081468,
   'raw_content': None},
  {'url': 'https://www.bbc.com/news/topics/cg41ylwvwgxt',
   'title': 'Venezuela - BBC News',
   'content': 'The US launched strikes on Venezuela on Saturday in which Maduro and his wife were captured by